# Shark survey batch pipeline

Runs the full chain for every flight **sequence** listed in `shark-data-info.ods`:

```
SRT + AirData + video clips
        |  generate_flat_surface_dem.py         (per-sequence flat DEM, sea level)
        v
  flat_surface_dem.json
        |  extract_video_frames.py               (frames + poses.json, no QGIS needed)
        v
   frames_w/poses.json
        |  trex_to_bambi.py                       (geo-reference the TRex tracklets)
        v
   tracks_w/tracks.csv  (+ detections_w, tracks_pixel_w, georeferenced_w)
```

Assumptions baked into this notebook (change in the config cell if wrong):

- All flights happened in the Maldives -> timezone `Indian/Maldives` for every sequence.
- No real bathymetric DEM is available for these sites -> every sequence uses a flat
  sea-level surface (`--flat-surface-msl 0.0`) generated fresh per sequence from its own
  GPS track (`generate_flat_surface_dem.py`).
- Camera is always the RGB/wide (`W`) camera.
- Drone (`pink`/`blue`) — and therefore which camera calibration to use — is inferred from
  which subfolder the sequence's AirData CSV lives in
  (`log-files/airdata/pink/...` vs `log-files/airdata/blue/...`).
- TRex tracklet variant: always the plain `*_id*.npz` files. `*_posture_id*.npz` is a
  *different* TRex export type (posture/midline shape analysis only) that doesn't carry
  the frame/pose-keypoint/detection fields this pipeline needs, so it's never used here.
  Anything containing `fschool` (fish-school tracking, a different task) is always excluded.
- A sequence is only processed if its `airdata-file` column is filled in in the spreadsheet.
  The ordered list of video clips is taken from `<sequence>.txt` (an ffmpeg-concat-style list
  of clip filenames) if that file exists next to the SRTs, otherwise the sequence is assumed
  to be a single clip named `<sequence>.MP4`.

Everything is idempotent: re-running a cell skips sequences whose output already exists.

In [8]:
import os

# cv2/numpy/BLAS default to spawning up to nproc threads, often with busy-spinning
# idle worker threads once initialized. This env block MUST run before pandas/numpy
# are imported (their thread pools latch onto these values on first use) - otherwise
# even this notebook's own kernel process ends up with a permanently bloated thread
# pool for its whole session, starving every subprocess we launch afterwards. If
# you've already run this notebook once without this cap, RESTART THE KERNEL (not
# just re-run cells) - an already-initialized thread pool can't be shrunk in-process.
THREAD_LIMIT = "4"
os.environ["OMP_NUM_THREADS"] = THREAD_LIMIT
os.environ["OPENBLAS_NUM_THREADS"] = THREAD_LIMIT
os.environ["MKL_NUM_THREADS"] = THREAD_LIMIT
os.environ["NUMEXPR_NUM_THREADS"] = THREAD_LIMIT
os.environ["OPENCV_NUM_THREADS"] = THREAD_LIMIT

import re
import shutil
import subprocess
import sys

import pandas as pd

# ── Repos ────────────────────────────────────────────────────────────────────
BAMBI_REPO = "/home/angela/Seafile/test-data/pink/code/bambi_detection"
TREX_REPO = "/home/angela/Seafile/test-data/pink/code/TRexConnector"

EXTRACT_SCRIPT = os.path.join(BAMBI_REPO, "src", "bambi", "extract_video_frames.py")
DEM_SCRIPT = os.path.join(BAMBI_REPO, "src", "bambi", "generate_flat_surface_dem.py")
TREX_SCRIPT = os.path.join(TREX_REPO, "trex_to_bambi.py")

# trex_to_bambi.py needs alfspy/bambi, which aren't installed in this notebook's kernel
# env - run it with the dedicated "trexconnector" conda env instead of sys.executable.
TREX_PYTHON = "/home/angela/miniforge3/envs/trexconnector/bin/python"

# ── Raw data ─────────────────────────────────────────────────────────────────
SHARKS_ROOT = "/media/angela/2026-drone/sharks_data"
VIDEOS_ROOT = "/media/maldives/2024_sequences/sequences-yolo26"
NPZ_DIR = os.path.join(VIDEOS_ROOT, "data")
SRT_DIR = os.path.join(SHARKS_ROOT, "log-files", "SRT")
CALIB_DIR = os.path.join(SHARKS_ROOT, "camera-calibration-data")
ODS_PATH = os.path.join(SHARKS_ROOT, "hand-timestamps", "shark-data-info.ods")

# ── Output ───────────────────────────────────────────────────────────────────
OUTPUT_ROOT = os.path.join(SHARKS_ROOT, "processed_data", "bambi_pipeline")
GEOREF_VIDEOS_DIR = os.path.join(SHARKS_ROOT, "georeferenced-videos")

# ── Fixed assumptions (see markdown above) ──────────────────────────────────
TIMEZONE = "Indian/Maldives"
CAMERA = "W"
FLAT_SURFACE_MSL = 0.0

# ── Debug mode ───────────────────────────────────────────────────────────────
# Set to an int (e.g. 200) to only extract/georeference/render the first N frames of
# each sequence - useful for a quick end-to-end test before committing to a full run.
# Debug output goes to a separate "<out_dir>_debug<N>" tree so it never collides with
# (or gets mistaken for) real full-run output. Set back to None for full runs.
# Requires the TimedFrameExtractorCallback early-stop fix in bambi's
# webgl/timed_pose_extractor.py - otherwise --limit doesn't actually skip decoding.
DEBUG_FRAME_LIMIT = None

# Set to a sequence name (e.g. "20240307_063012765_DJI_0463") to only process that one
# sequence in every step below - handy combined with DEBUG_FRAME_LIMIT for a fast,
# single-sequence smoke test. Set to None to process every ready sequence.
# DEBUG_SEQUENCE = "sequence_20240303_060726300_DJI_0242"
DEBUG_SEQUENCE = None

def entry_out_dir(entry):
    """Per-sequence output dir, redirected to a debug-specific tree when
    DEBUG_FRAME_LIMIT is set (see 'Debug mode' above)."""
    if DEBUG_FRAME_LIMIT is None:
        return entry["out_dir"]
    return f"{entry['out_dir']}_debug{DEBUG_FRAME_LIMIT}"


def should_process(entry):
    """Whether a manifest entry should be processed by the steps below, honouring
    both readiness and the optional DEBUG_SEQUENCE filter."""
    if not entry["ready"]:
        return False
    if DEBUG_SEQUENCE is not None and entry["sequence"] != DEBUG_SEQUENCE:
        return False
    return True


def run_capped(cmd):
    """subprocess.run with the thread cap above and check=True."""
    subprocess.run(cmd, check=True, env=os.environ)


os.makedirs(OUTPUT_ROOT, exist_ok=True)
os.makedirs(GEOREF_VIDEOS_DIR, exist_ok=True)

## Step 0 — Build the manifest

Reads the `overview` sheet, keeps sequences that have an `airdata-file`, and resolves every
path the rest of the pipeline needs. Nothing is executed on the *video/tracking* data yet -
this only inspects the filesystem so problems (missing clips, no matching tracklets, ...)
show up before any processing starts.

One exception: for multi-clip sequences that only have a pre-concatenated single video left
on disk (the 9 raw per-clip files having been deleted), DJI still only wrote one `.SRT` per
raw clip, while `extract_video_frames.py` requires exactly one SRT per video file. So this
step builds (and caches next to the raw SRTs, as `<sequence>_merged.SRT`) a single merged SRT
from the raw per-clip SRTs, using `bambi.srt.SrtMerger`. This was verified to line up exactly,
frame-for-frame, with the merged video (merged SRT frame count == video frame count, and the
embedded telemetry timestamps are continuous with no gap across clip boundaries).

In [9]:
def _parse_concat_list(path):
    """Parse an ffmpeg-concat-style 'file <name>' list into an ordered filename list."""
    names = []
    with open(path) as f:
        for line in f:
            line = line.strip()
            if line.startswith("file "):
                names.append(line[len("file "):].strip().strip("'\""))
    return names


def _resolve_drone(airdata_path):
    if f"{os.sep}airdata{os.sep}pink{os.sep}" in airdata_path:
        return "pink"
    if f"{os.sep}airdata{os.sep}blue{os.sep}" in airdata_path:
        return "blue"
    return None


def _resolve_npz(seq):
    """Pick the tracklet npz files for a sequence: the plain *_id*.npz files, always
    excluding anything with 'fschool' in the name.

    *_posture_id*.npz is a *different* TRex export type (posture/midline shape
    analysis only - fields like "frames", "midline_lengths", "hole_points") and does
    NOT carry the frame/pose-keypoint/detection_class fields trex_to_bambi.py needs -
    it is not a richer/preferred version of the plain tracklet data, so it must never
    be used as the tracklet source here.
    """
    if not os.path.isdir(NPZ_DIR):
        return None, []
    candidates = [
        f for f in os.listdir(NPZ_DIR)
        if f.startswith(seq + "_") and f.endswith(".npz")
        and "fschool" not in f and "_posture_id" not in f
    ]
    if candidates:
        return "plain", sorted(candidates)
    return None, []


def _build_merged_srt(seq, raw_srt_names):
    """Merge a multi-clip sequence's raw per-clip SRTs into one SRT matching the
    already-concatenated video. extract_video_frames.py requires exactly one SRT per
    video file; DJI writes one SRT per raw clip instead. Cached next to the raw SRTs
    as "<sequence>_merged.SRT" (all drones/cameras in this survey are DJI M30T Wide).
    """
    from bambi.domain.camera import Camera
    from bambi.domain.drone import Drone
    from bambi.srt.srt_merger import SrtMerger
    from bambi.srt.srt_parser import SrtParser
    from bambi.srt.srt_writer import SrtWriter

    merged_path = os.path.join(SRT_DIR, f"{seq}_merged.SRT")
    if os.path.isfile(merged_path):
        return merged_path

    parser = SrtParser()
    frame_lists = [parser.parse(os.path.join(SRT_DIR, name)) for name in raw_srt_names]
    merged_frames = SrtMerger().merge(frame_lists)
    SrtWriter().write(merged_path, merged_frames, Drone.M30T, Camera.Wide)
    print(f"[run]  {seq}: merged {len(raw_srt_names)} per-clip SRTs -> {os.path.basename(merged_path)}")
    return merged_path


def _resolve_video_and_srts(seq):
    """Pick the video file(s) and matching SRT file(s) for a sequence.

    Prefer an already-concatenated single video matching the sequence name exactly
    (e.g. "sequence_<id>.mp4") over the raw per-clip files listed in the sequence's
    concat .txt list: TRex was run directly against that merged file (confirmed via
    its .settings "meta_source_path"), and for many sequences the raw per-clip files
    no longer exist on disk at all - only the merged file remains.

    When using the merged video and the concat .txt lists more than one raw clip, the
    matching SRT is built via _build_merged_srt from those clips' raw per-clip SRTs.
    Otherwise (a genuinely single-clip sequence, or no concat .txt) falls back to the
    plain per-clip-name-based SRT guess - no merge needed for a single file.
    """
    concat_txt = os.path.join(SRT_DIR, f"{seq}.txt")
    raw_clip_names = _parse_concat_list(concat_txt) if os.path.isfile(concat_txt) else None

    for ext in (".mp4", ".MP4"):
        single_video = os.path.join(VIDEOS_ROOT, f"{seq}{ext}")
        if os.path.isfile(single_video):
            videos = [single_video]
            if raw_clip_names and len(raw_clip_names) > 1:
                raw_srt_names = [os.path.splitext(n)[0] + ".SRT" for n in raw_clip_names]
                if all(os.path.isfile(os.path.join(SRT_DIR, n)) for n in raw_srt_names):
                    return videos, [_build_merged_srt(seq, raw_srt_names)]
            srt_guess = os.path.join(SRT_DIR, os.path.splitext(os.path.basename(single_video))[0] + ".SRT")
            return videos, [srt_guess]

    clip_names = raw_clip_names or [f"{seq}.MP4"]
    videos = [os.path.join(VIDEOS_ROOT, name) for name in clip_names]
    srts = [os.path.join(SRT_DIR, os.path.splitext(name)[0] + ".SRT") for name in clip_names]
    return videos, srts


def resolve_sequence(row):
    seq = row["Sequence"]
    airdata = row["airdata-file"]
    if pd.isna(seq) or pd.isna(airdata):
        return None

    drone = _resolve_drone(airdata)
    videos, srts = _resolve_video_and_srts(seq)
    calib_path = os.path.join(CALIB_DIR, f"{drone}_drone_combined.json") if drone else None
    npz_variant, npz_names = _resolve_npz(seq)

    missing_videos = [v for v in videos if not os.path.isfile(v)]
    missing_srts = [s for s in srts if not os.path.isfile(s)]
    missing_reasons = []
    if drone is None:
        missing_reasons.append("unrecognised drone (airdata not under pink/ or blue/)")
    if missing_videos:
        missing_reasons.append(f"{len(missing_videos)} missing video clip(s)")
    if missing_srts:
        missing_reasons.append(f"{len(missing_srts)} missing SRT file(s)")
    if npz_variant is None:
        missing_reasons.append("no matching (non-fschool) TRex tracklets")
    if calib_path and not os.path.isfile(calib_path):
        missing_reasons.append(f"calibration file not found: {calib_path}")

    return {
        "sequence": seq,
        "drone": drone,
        "airdata": airdata,
        "videos": videos,
        "srts": srts,
        "num_clips": len(videos),
        "calib_path": calib_path,
        "npz_variant": npz_variant,
        "npz_paths": [os.path.join(NPZ_DIR, n) for n in npz_names],
        "ready": not missing_reasons,
        "issues": "; ".join(missing_reasons),
        "out_dir": os.path.join(OUTPUT_ROOT, seq),
    }


overview = pd.read_excel(ODS_PATH, engine="odf", sheet_name="overview")
manifest = [resolve_sequence(row) for _, row in overview.iterrows()]
manifest = [m for m in manifest if m is not None]
print(f"{len(manifest)} sequences have an airdata-file; "
      f"{sum(m['ready'] for m in manifest)} are ready to process.")

11 sequences have an airdata-file; 11 are ready to process.


In [10]:
pd.DataFrame(manifest)[["sequence", "drone", "num_clips", "npz_variant", "ready", "issues"]]

,sequence,drone,num_clips,npz_variant,ready,issues
0,sequence_20240303_070126703_DJI_0257,blue,1,plain,True,
1,sequence_20240303_060726300_DJI_0242,blue,1,plain,True,
2,sequence_20240303_072831125_DJI_0266,blue,1,plain,True,
3,sequence_20240305_070907124_DJI_0266,pink,1,plain,True,
4,sequence_20240305_164039273_DJI_0305,pink,1,plain,True,
5,sequence_20240306_161443984_DJI_0139,blue,1,plain,True,
6,sequence_20240305_063615511_DJI_0253,pink,1,plain,True,
7,sequence_20240308_074843321_DJI_0502,blue,1,plain,True,
8,sequence_20240302_083729249_DJI_0001,pink,1,plain,True,
9,sequence_20240307_070355688_DJI_0202,blue,1,plain,True,


## Step 1 — Generate a flat-surface DEM per sequence

Uses the sequence's own AirData CSV to compute a GPS-centred flat surface at sea level
(`--elevation 0.0`); the UTM zone is auto-detected per sequence. Skips sequences that already
have a `flat_surface_dem.json`.

In [11]:
def generate_dem(entry):
    dem_dir = os.path.join(entry["out_dir"], "dem")
    dem_json = os.path.join(dem_dir, "flat_surface_dem.json")
    if os.path.isfile(dem_json):
        print(f"[skip] {entry['sequence']}: DEM already exists")
        return dem_json
    os.makedirs(dem_dir, exist_ok=True)
    cmd = [
        sys.executable, DEM_SCRIPT,
        "--inputs", entry["airdata"],
        "--elevation", str(FLAT_SURFACE_MSL),
        "--output", dem_dir,
        "--name", "flat_surface_dem",
    ]
    print(f"[run]  {entry['sequence']}: generating flat-surface DEM")
    run_capped(cmd)
    return dem_json


for entry in manifest:
    if should_process(entry):
        entry["dem_json"] = generate_dem(entry)

[skip] sequence_20240303_070126703_DJI_0257: DEM already exists
[skip] sequence_20240303_060726300_DJI_0242: DEM already exists
[skip] sequence_20240303_072831125_DJI_0266: DEM already exists
[skip] sequence_20240305_070907124_DJI_0266: DEM already exists
[skip] sequence_20240305_164039273_DJI_0305: DEM already exists
[skip] sequence_20240306_161443984_DJI_0139: DEM already exists
[skip] sequence_20240305_063615511_DJI_0253: DEM already exists
[skip] sequence_20240308_074843321_DJI_0502: DEM already exists
[skip] sequence_20240302_083729249_DJI_0001: DEM already exists
[skip] sequence_20240307_070355688_DJI_0202: DEM already exists
[skip] 20240307_063012765_DJI_0463: DEM already exists


## Step 2 — Extract frames + poses

Runs `extract_video_frames.py` for each sequence (all its clips, in concat order), anchored
to the DEM origin from step 1. Skips sequences that already have a `poses.json`.

In [ ]:
def extract_frames(entry):
    frames_dir = os.path.join(entry_out_dir(entry), "frames_w")
    poses_json = os.path.join(frames_dir, "poses.json")
    if os.path.isfile(poses_json):
        print(f"[skip] {entry['sequence']}: frames already extracted")
        return frames_dir
    cmd = [
        sys.executable, EXTRACT_SCRIPT,
        "--videos", *entry["videos"],
        "--srts", *entry["srts"],
        "--airdata", entry["airdata"],
        "--camera", CAMERA,
        "--calibration-path", entry["calib_path"],
        "--dem-json", entry["dem_json"],
        "--output", frames_dir,
        "--timezone", TIMEZONE,
    ]
    if DEBUG_FRAME_LIMIT is not None:
        cmd += ["--limit", str(DEBUG_FRAME_LIMIT)]
    print(f"[run]  {entry['sequence']}: extracting {entry['num_clips']} clip(s)"
          + (f" (debug: first {DEBUG_FRAME_LIMIT} frames)" if DEBUG_FRAME_LIMIT is not None else ""))
    run_capped(cmd)
    return frames_dir


for entry in manifest:
    if should_process(entry):
        entry["frames_dir"] = extract_frames(entry)

[run]  sequence_20240303_070126703_DJI_0257: extracting 1 clip(s)


## Step 3 — Geo-reference the TRex tracklets

The plain tracklets (`*_id*.npz`, never `_posture_id` or `fschool`) are copied into a
per-sequence staging folder, since `trex_to_bambi.py` reads *every* `.npz` in its
`--npz-dir` and the real tracklets for all sequences live together in one shared folder.
(Copied rather than symlinked because the output drive is exFAT, which doesn't support
symlinks.) Skips sequences that already have a `tracks.csv`.

In [ ]:
def stage_npz(entry):
    """Copy this sequence's tracklet npz files into a per-sequence staging folder
    (trex_to_bambi.py reads *every* .npz in --npz-dir). Syncs rather than just
    adding: also removes any stale .npz left over from a previous run whose file set
    doesn't match the current entry["npz_paths"] (e.g. after _resolve_npz's logic
    changed, or a debug re-run) - trex_to_bambi.py has no way to filter unwanted
    files itself, so a stale leftover silently corrupts the read.
    """
    staging = os.path.join(entry_out_dir(entry), "npz_input")
    os.makedirs(staging, exist_ok=True)
    wanted = {os.path.basename(src) for src in entry["npz_paths"]}
    for existing in os.listdir(staging):
        if existing.endswith(".npz") and existing not in wanted:
            os.remove(os.path.join(staging, existing))
    for src in entry["npz_paths"]:W
        dst = os.path.join(staging, os.path.basename(src))
        if not os.path.exists(dst):
            shutil.copy2(src, dst)
    return staging


def run_trex_connector(entry):
    out_dir = entry_out_dir(entry)
    tracks_csv = os.path.join(out_dir, "tracks_w", "tracks.csv")
    if os.path.isfile(tracks_csv):
        print(f"[skip] {entry['sequence']}: already geo-referenced")
        return
    npz_dir = stage_npz(entry)
    mask_path = os.path.join(entry["frames_dir"], f"mask_{CAMERA}.png")
    cmd = [
        TREX_PYTHON, TREX_SCRIPT,
        "--npz-dir", npz_dir,
        "--dem-json", entry["dem_json"],
        "--poses", os.path.join(entry["frames_dir"], "poses.json"),
        "--calib", entry["calib_path"],
        "--mask", mask_path,
        "--out-dir", out_dir,
        "--flat-surface-msl", str(FLAT_SURFACE_MSL),
    ]
    print(f"[run]  {entry['sequence']}: geo-referencing ({entry['npz_variant']} tracklets)")
    run_capped(cmd)


for entry in manifest:
    if should_process(entry):
        run_trex_connector(entry)

[run]  sequence_20240303_060726300_DJI_0242: geo-referencing (plain tracklets)
1. Reading TRex tracklets
  sequence_20240303_060726300_DJI_0242_id0.npz: track 0, 17848 detections
  sequence_20240303_060726300_DJI_0242_id1.npz: track 1, 4335 detections
  sequence_20240303_060726300_DJI_0242_id2.npz: track 2, 14549 detections
  sequence_20240303_060726300_DJI_0242_id3.npz: track 3, 14950 detections
  sequence_20240303_060726300_DJI_0242_id4.npz: track 4, 15284 detections
  sequence_20240303_060726300_DJI_0242_id5.npz: track 5, 18343 detections
  sequence_20240303_060726300_DJI_0242_id6.npz: track 6, 27607 detections
   112916 detections, raw video size (5120, 2700)
Wrote 112916 detections -> /media/angela/2026-drone/sharks_data/processed_data/bambi_pipeline/sequence_20240303_060726300_DJI_0242_debug1000/detections_w/detections.txt
Wrote 112916 pixel-space track rows -> /media/angela/2026-drone/sharks_data/processed_data/bambi_pipeline/sequence_20240303_060726300_DJI_0242_debug1000/tracks

## Step 4 — Render georeferenced videos

Renders the side-by-side pixel-space / geo-space video for every `ready` sequence (full
length, no frame cap) and writes each one to `GEOREF_VIDEOS_DIR` as
`<sequence>_trex_vis.mp4`. Skips sequences whose output video already exists.

Note: `--calib` is intentionally *not* passed here — `--video` is the original raw video,
and `--calib` would undistort the keypoints into a space the raw video isn't in, misaligning
them. Only use `--calib` when `--video` points at the undistorted extracted frames instead.

In [ ]:
def render_georeferenced_video(entry):
    suffix = f"_debug{DEBUG_FRAME_LIMIT}" if DEBUG_FRAME_LIMIT is not None else ""
    out_path = os.path.join(GEOREF_VIDEOS_DIR, f"{entry['sequence']}_trex_vis{suffix}.mp4")
    if os.path.isfile(out_path):
        print(f"[skip] {entry['sequence']}: video already exists")
        return out_path
    cmd = [
        sys.executable, os.path.join(TREX_REPO, "visualize_trex_video_and_map.py"),
        "--video", entry["videos"][0],
        "--tracking-dir", NPZ_DIR,
        "--tracks-csv", os.path.join(entry_out_dir(entry), "tracks_w", "tracks.csv"),
        "--poses", os.path.join(entry["frames_dir"], "poses.json"),
        "--dem-json", entry["dem_json"],
        "--output", out_path,
        # No --calib here: --video is the original raw video, and --calib would
        # undistort the keypoints into a space the raw video isn't in, misaligning
        # them. Only pass --calib when --video is the undistorted extracted frames.
        "--no-live",
    ]
    if DEBUG_FRAME_LIMIT is not None:
        cmd += ["--max-frames", str(DEBUG_FRAME_LIMIT)]
    print(f"[run]  {entry['sequence']}: rendering georeferenced video"
          + (f" (debug: first {DEBUG_FRAME_LIMIT} frames)" if DEBUG_FRAME_LIMIT is not None else ""))
    run_capped(cmd)
    return out_path


for entry in manifest:
    if should_process(entry):
        render_georeferenced_video(entry)

[run]  sequence_20240303_060726300_DJI_0242: rendering georeferenced video (debug: first 1000 frames)
Video      : /media/maldives/2024_sequences/sequences-yolo26/sequence_20240303_060726300_DJI_0242.mp4  (5120x2700 @ 50.0 fps)
TRex       : 7 tracklets, detections on 32424 frames
Geo tracks : 1000 frames, extent E[257440.4,257448.0] N[430063.0,430068.7]
Drone      : loading poses ...
Drone      : 1000 positions
FFMPEG writer unavailable ([Errno 2] No such file or directory: 'ffmpeg -version'); using cv2.VideoWriter fallback.
Writing video -> /media/angela/2026-drone/sharks_data/georeferenced-videos/sequence_20240303_060726300_DJI_0242_trex_vis_debug1000.mp4
Done: /media/angela/2026-drone/sharks_data/georeferenced-videos/sequence_20240303_060726300_DJI_0242_trex_vis_debug1000.mp4
